# Stage 2 — Character sheets, shot images, and clips

Runs on Kaggle (Settings → Accelerator → GPU T4 x2 or P100).

**Before running:** add these as Kaggle Secrets (Add-ons → Secrets):
- `HF_TOKEN` — Hugging Face token (write access) — used as your free asset store
- `HF_REPO` — e.g. `yourname/video-pipeline-assets` (a dataset repo you created)

**Set `PROJECT_ID` below** to the folder name Stage 1 printed out.

What this notebook does, in order:
1. Installs ComfyUI + IP-Adapter + LTX-Video nodes
2. Downloads SDXL + IP-Adapter + LTX-Video weights (cached after first run)
3. Pulls the project manifest from your HF repo
4. For each character: generates a reference sheet (front/side/3-4 view, neutral)
5. For each shot: generates the frame, conditioned on the reference sheet(s) of the
   characters present, via IP-Adapter — this is what keeps faces/colors consistent
   across shots instead of re-rolling a new-looking character every time
6. Feeds each selected frame into LTX-Video (image-to-video) to get a short clip
7. Pushes everything back to the HF repo and updates manifest.json

Session limit is 12h / ~30 GPU-hrs per week. If you run out mid-project, just
re-run this notebook later — it skips any shot whose manifest status is already `done`.

In [ ]:
PROJECT_ID = "project_042"  # <-- change this to your Stage 1 output folder name

In [ ]:
# --- 1. Install ComfyUI + custom nodes ---
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI
%cd /kaggle/working/ComfyUI
!pip install -q -r requirements.txt

!mkdir -p custom_nodes
%cd custom_nodes
!git clone --depth 1 https://github.com/cubiq/ComfyUI_IPAdapter_plus.git
!git clone --depth 1 https://github.com/Lightricks/ComfyUI-LTXVideo.git
%cd /kaggle/working/ComfyUI
!pip install -q huggingface_hub

In [ ]:
# --- 2. Download weights (cached in Kaggle's /kaggle/working across the session) ---
from huggingface_hub import hf_hub_download
import os

os.makedirs("models/checkpoints", exist_ok=True)
os.makedirs("models/ipadapter", exist_ok=True)
os.makedirs("models/clip_vision", exist_ok=True)
os.makedirs("models/checkpoints/ltx", exist_ok=True)

# SDXL base checkpoint
hf_hub_download("stabilityai/stable-diffusion-xl-base-1.0",
                 "sd_xl_base_1.0.safetensors", local_dir="models/checkpoints")

# IP-Adapter (this is what conditions new images on the character reference sheet)
hf_hub_download("h94/IP-Adapter", "sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors",
                 local_dir="models/ipadapter")
hf_hub_download("h94/IP-Adapter", "models/image_encoder/model.safetensors",
                 local_dir="models/clip_vision")

# LTX-Video (image-to-video)
hf_hub_download("Lightricks/LTX-Video", "ltx-video-2b-v0.9.safetensors",
                 local_dir="models/checkpoints/ltx")

print("Weights ready.")

In [ ]:
# --- 3. Pull the project manifest from your HF dataset repo ---
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download, upload_file
import json, shutil

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
HF_REPO = secrets.get_secret("HF_REPO")

api = HfApi(token=HF_TOKEN)
local_project = f"/kaggle/working/{PROJECT_ID}"

api.snapshot_download(repo_id=HF_REPO, repo_type="dataset",
                       allow_patterns=[f"{PROJECT_ID}/*", f"{PROJECT_ID}/**"],
                       local_dir="/kaggle/working/")

manifest_path = f"{local_project}/manifest.json"
manifest = json.load(open(manifest_path))
print(f"Loaded {PROJECT_ID}: {len(manifest['characters'])} characters, "
      f"{len(manifest['shots'])} shots")

def save_manifest():
    json.dump(manifest, open(manifest_path, "w"), indent=2)
    api.upload_file(path_or_fileobj=manifest_path,
                     path_in_repo=f"{PROJECT_ID}/manifest.json",
                     repo_id=HF_REPO, repo_type="dataset")

In [ ]:
# --- 4. ComfyUI helper: run a workflow graph via the API ---
# Start the ComfyUI server in the background, then POST workflow JSON to it.
import subprocess, time, urllib.request, json, uuid, websocket

server_proc = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/kaggle/working/ComfyUI"
)
time.sleep(20)  # give it time to load
print("ComfyUI server started on :8188")

def queue_prompt(workflow: dict) -> str:
    client_id = str(uuid.uuid4())
    payload = json.dumps({"prompt": workflow, "client_id": client_id}).encode()
    req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=payload)
    resp = json.loads(urllib.request.urlopen(req).read())
    prompt_id = resp["prompt_id"]

    ws = websocket.WebSocket()
    ws.connect(f"ws://127.0.0.1:8188/ws?clientId={client_id}")
    while True:
        msg = json.loads(ws.recv())
        if msg.get("type") == "executing":
            data = msg["data"]
            if data.get("node") is None and data.get("prompt_id") == prompt_id:
                break
    ws.close()
    return prompt_id

In [ ]:
# --- 5. Character reference sheets (generated once per character, no IP-Adapter needed) ---
# 4-panel sheet: front / 3-quarter / side / neutral close-up, same seed family for coherence

def build_sheet_workflow(description: str, seed: int, out_prefix: str) -> dict:
    prompt_text = (
        f"character reference sheet, {description}, turnaround sheet, "
        f"front view, side view, three-quarter view, neutral expression, "
        f"plain grey background, consistent design, full body, clean line art shading, "
        f"high detail, concept art"
    )
    # Minimal SDXL txt2img graph — trimmed to the essential nodes.
    return {
        "3": {"class_type": "KSampler", "inputs": {
            "seed": seed, "steps": 30, "cfg": 7.0, "sampler_name": "dpmpp_2m",
            "scheduler": "karras", "denoise": 1.0,
            "model": ["4", 0], "positive": ["6", 0], "negative": ["7", 0],
            "latent_image": ["5", 0]}},
        "4": {"class_type": "CheckpointLoaderSimple",
              "inputs": {"ckpt_name": "sd_xl_base_1.0.safetensors"}},
        "5": {"class_type": "EmptyLatentImage",
              "inputs": {"width": 1024, "height": 1024, "batch_size": 1}},
        "6": {"class_type": "CLIPTextEncode",
              "inputs": {"text": prompt_text, "clip": ["4", 1]}},
        "7": {"class_type": "CLIPTextEncode",
              "inputs": {"text": "blurry, extra limbs, deformed, watermark, text",
                         "clip": ["4", 1]}},
        "8": {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["4", 2]}},
        "9": {"class_type": "SaveImage",
              "inputs": {"filename_prefix": out_prefix, "images": ["8", 0]}},
    }

for char in manifest["characters"]:
    if char["status"] == "done":
        print(f"skip {char['id']} (already done)")
        continue
    seed = abs(hash(char["id"])) % (2**31)
    wf = build_sheet_workflow(char["description"], seed, f"{char['id']}_sheet")
    queue_prompt(wf)
    char["status"] = "generated_pending_review"
    print(f"generated sheet for {char['name']}")

save_manifest()
print("\nCharacter sheets are in ComfyUI/output/. Eyeball them before continuing --\n"
      "if one looks wrong, tweak the description in manifest.json and re-run this cell\n"
      "for just that character.")

In [ ]:
# --- 6. Shot images, conditioned on character reference sheet(s) via IP-Adapter ---
# This is the step that actually keeps characters consistent shot-to-shot: instead of
# describing the character in text every time (which drifts), we feed the model the
# reference image itself as a conditioning signal.

def build_shot_workflow(shot: dict, ref_image_paths: list, seed: int, out_prefix: str) -> dict:
    graph = {
        "4": {"class_type": "CheckpointLoaderSimple",
              "inputs": {"ckpt_name": "sd_xl_base_1.0.safetensors"}},
        "5": {"class_type": "EmptyLatentImage",
              "inputs": {"width": 1280, "height": 720, "batch_size": 1}},
        "6": {"class_type": "CLIPTextEncode",
              "inputs": {"text": shot["description"], "clip": ["4", 1]}},
        "7": {"class_type": "CLIPTextEncode",
              "inputs": {"text": "blurry, deformed, watermark, text, inconsistent design",
                         "clip": ["4", 1]}},
        "10": {"class_type": "IPAdapterUnifiedLoader",
               "inputs": {"model": ["4", 0], "preset": "PLUS (high strength)"}},
    }
    # chain one IPAdapter apply node per reference image (multi-character shots)
    prev_model = ["10", 0]
    for i, ref_path in enumerate(ref_image_paths):
        load_id = f"load_ref_{i}"
        apply_id = f"apply_ip_{i}"
        graph[load_id] = {"class_type": "LoadImage", "inputs": {"image": ref_path}}
        graph[apply_id] = {"class_type": "IPAdapterAdvanced", "inputs": {
            "model": prev_model, "ipadapter": ["10", 1],
            "image": [load_id, 0], "weight": 0.75, "weight_type": "linear",
            "combine_embeds": "average", "start_at": 0.0, "end_at": 1.0}}
        prev_model = [apply_id, 0]

    graph["3"] = {"class_type": "KSampler", "inputs": {
        "seed": seed, "steps": 30, "cfg": 6.5, "sampler_name": "dpmpp_2m",
        "scheduler": "karras", "denoise": 1.0,
        "model": prev_model, "positive": ["6", 0], "negative": ["7", 0],
        "latent_image": ["5", 0]}}
    graph["8"] = {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["4", 2]}}
    graph["9"] = {"class_type": "SaveImage",
                   "inputs": {"filename_prefix": out_prefix, "images": ["8", 0]}}
    return graph

char_by_id = {c["id"]: c for c in manifest["characters"]}

for shot in manifest["shots"]:
    if shot["image"]["status"] == "done":
        print(f"skip {shot['shot_id']} (already done)")
        continue
    ref_paths = [char_by_id[cid]["reference_sheet"] for cid in shot["characters_present"]]
    seed = abs(hash(shot["shot_id"])) % (2**31)
    wf = build_shot_workflow(shot, ref_paths, seed, shot["shot_id"])
    queue_prompt(wf)
    shot["image"]["status"] = "done"
    print(f"generated frame for {shot['shot_id']}")

save_manifest()

In [ ]:
# --- 7. Image-to-video with LTX-Video ---
# Always image-to-video (never text-to-video) -- starting from the already-consistent
# frame is what stops the character's face/colors drifting mid-clip.

def build_ltx_workflow(image_path: str, duration_s: int, seed: int, out_prefix: str) -> dict:
    fps = 24
    num_frames = duration_s * fps
    return {
        "1": {"class_type": "LoadImage", "inputs": {"image": image_path}},
        "2": {"class_type": "LTXVLoader",
              "inputs": {"ckpt_name": "ltx/ltx-video-2b-v0.9.safetensors"}},
        "3": {"class_type": "LTXVImgToVideo", "inputs": {
            "model": ["2", 0], "image": ["1", 0],
            "width": 768, "height": 512, "num_frames": num_frames,
            "seed": seed, "steps": 25, "cfg": 3.0}},
        "4": {"class_type": "VHS_VideoCombine", "inputs": {
            "images": ["3", 0], "frame_rate": fps, "filename_prefix": out_prefix,
            "format": "video/h264-mp4"}},
    }

for shot in manifest["shots"]:
    if shot["video"]["status"] == "done":
        print(f"skip {shot['shot_id']} video (already done)")
        continue
    selected_frame = shot["image"].get("selected_variant") or shot["image"]["path"]
    seed = abs(hash(shot["shot_id"] + "_vid")) % (2**31)
    t0 = time.time()
    wf = build_ltx_workflow(selected_frame, shot["duration_seconds"], seed, shot["shot_id"] + "_clip")
    queue_prompt(wf)
    shot["video"]["status"] = "done"
    shot["video"]["model"] = "ltx-video"
    shot["video"]["render_seconds"] = round(time.time() - t0, 1)
    print(f"rendered clip for {shot['shot_id']} in {shot['video']['render_seconds']}s")

save_manifest()

In [ ]:
# --- 8. Push everything (images + clips) back to the HF repo ---
import glob

for f in glob.glob("/kaggle/working/ComfyUI/output/*"):
    fname = os.path.basename(f)
    api.upload_file(path_or_fileobj=f,
                     path_in_repo=f"{PROJECT_ID}/generated/{fname}",
                     repo_id=HF_REPO, repo_type="dataset")

manifest["status"] = "rendered_pending_assembly"
save_manifest()
print("Done. Hand off to Stage 3 (assembly) on the Oracle box / GitHub Actions runner.")